# 実行環境の確認
モデル・データをダウンロードせず、CPUでDE/BEMAと小さいLlama/GemmaのLoRA逆伝播を検証します。実モデルの精度再現テストではありません。

In [1]:
from pathlib import Path
import sys
candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/RMT_utils')]
ROOT = next(p for p in candidates if (p / 'notebook_runtime.py').is_file())
sys.path.insert(0, str(ROOT))
import torch, numpy as np, transformers, peft
import funcs1
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'cuda': torch.cuda.is_available()})


/Users/ryoyazushi/GitHub/RMT_utils/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.11.13', 'torch': '2.8.0', 'transformers': '4.57.6', 'peft': '0.17.1', 'cuda': False}


In [2]:
rng = np.random.default_rng(42)
Y = rng.normal(size=(32, 64)) * np.sqrt(rng.uniform(.2, 3, (32, 1)))
Y_hat, x, y = funcs1.dyson_equalizer_algorithm1(Y)
assert Y_hat.shape == Y.shape and np.isfinite(Y_hat).all()
result = funcs1.bema_algorithm1_from_data(Y_hat)
assert np.isfinite(result['sigma2_hat'])
print({k: result[k] for k in ['sigma2_hat', 's_hat', 'threshold']})


{'sigma2_hat': np.float64(1.1364990091817797), 's_hat': 0, 'threshold': np.float64(3.3852013476938034)}


In [3]:
from transformers import LlamaConfig, LlamaForCausalLM, Gemma3TextConfig, Gemma3ForCausalLM
from peft import LoraConfig, TaskType, get_peft_model
for cls, config in [
    (LlamaForCausalLM, LlamaConfig(vocab_size=32, hidden_size=16, intermediate_size=32, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=1)),
    (Gemma3ForCausalLM, Gemma3TextConfig(vocab_size=32, hidden_size=16, intermediate_size=32, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=1, head_dim=8)),
]:
    torch.manual_seed(42)
    model = get_peft_model(cls(config), LoraConfig(task_type=TaskType.CAUSAL_LM, r=2, lora_alpha=4, target_modules=['q_proj', 'v_proj']))
    ids = torch.tensor([[1, 2, 3, 4]])
    loss = model(input_ids=ids, labels=ids).loss
    loss.backward()
    assert torch.isfinite(loss)
    print(cls.__name__, 'LoRA backward OK', float(loss.detach()))
    del model


LlamaForCausalLM LoRA backward OK 3.4194815158843994
Gemma3ForCausalLM LoRA backward OK 3.528355598449707
